In [ ]:
from pathlib import Path
import joblib

# Locate the NovaBank project folder.
project_root = Path.cwd()

while not (
    project_root / "models" / "xgb_temporal_pipeline.joblib"
).exists():
    if project_root.parent == project_root:
        raise FileNotFoundError(
            "Saved XGBoost model not found. "
            "Follow Step 2 below to save it first."
        )
    project_root = project_root.parent

# Load the saved model.
model_path = (
    project_root
    / "models"
    / "xgb_temporal_pipeline.joblib"
)

xgb_model = joblib.load(model_path)

# Get the fitted preprocessing pipeline and classifier.
fitted_preprocessor = xgb_model.named_steps["preprocess"]
fitted_xgb = xgb_model.named_steps["classifier"]

# Prepare the transformed feature names.
feature_names = fitted_preprocessor.get_feature_names_out()

print("XGBoost pipeline loaded successfully!")
print("Model path:", model_path)
print("Number of transformed features:", len(feature_names))

In [ ]:
required_variables = [
    "xgb_model",
    "X_train_time",
    "X_test_time",
    "numeric_features",
    "categorical_features",
]

for variable in required_variables:
    print(f"{variable}: {variable in globals()}")

In [ ]:
import pandas as pd
import numpy as np

# Load the prepared dataset.
DATA_FILE = PROJECT_ROOT / "data" / "processed" / "credit_risk_cohort.csv"

credit = pd.read_csv(DATA_FILE)

credit["issue_date"] = pd.to_datetime(
    credit["issue_date"], errors="coerce"
)

credit = credit.dropna(
    subset=["issue_date", "bad_outcome"]
).copy()

credit["bad_outcome"] = credit["bad_outcome"].astype(int)
credit["issue_month"] = credit["issue_date"].dt.to_period("M")


# Recover the exact input columns from the saved model.
fitted_preprocessor = xgb_model.named_steps["preprocess"]
fitted_xgb = xgb_model.named_steps["classifier"]

feature_columns = list(fitted_preprocessor.feature_names_in_)

numeric_features = list(
    fitted_preprocessor.transformers_[0][2]
)

categorical_features = list(
    fitted_preprocessor.transformers_[1][2]
)


# Recreate the chronological split used for model evaluation.
months = sorted(credit["issue_month"].dropna().unique())

split_position = int(len(months) * 0.80)
split_position = min(
    max(split_position, 1),
    len(months) - 1
)

cutoff_month = months[split_position]

train = credit[
    credit["issue_month"] < cutoff_month
].copy()

test = credit[
    credit["issue_month"] >= cutoff_month
].copy()


# Prepare the same features in the same order as model training.
for column in numeric_features:
    credit[column] = pd.to_numeric(
        credit[column], errors="coerce"
    )

# Rebuild the feature matrices after numeric conversion.
train = credit[
    credit["issue_month"] < cutoff_month
].copy()

test = credit[
    credit["issue_month"] >= cutoff_month
].copy()

X_train_time = train[feature_columns].copy()
X_test_time = test[feature_columns].copy()

y_train_time = train["bad_outcome"].astype(int)
y_test_time = test["bad_outcome"].astype(int)


# Confirm that the recreated split matches our earlier results.
print("Training rows:", len(X_train_time))
print("Test rows:", len(X_test_time))
print("Temporal cutoff:", cutoff_month)
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

assert list(X_train_time.columns) == list(
    fitted_preprocessor.feature_names_in_
)

assert len(X_train_time) == 19441
assert len(X_test_time) == 19136

print("\nData recreated successfully.")

In [ ]:
import numpy as np
import pandas as pd
import shap

from scipy import sparse

# 1. Get the fitted XGBoost model and preprocessor.
fitted_preprocessor = xgb_model.named_steps["preprocess"]
fitted_xgb = xgb_model.named_steps["classifier"]

# 2. Get feature names after preprocessing.
feature_names = fitted_preprocessor.get_feature_names_out()

# 3. Select reproducible samples from training and test data.
rng = np.random.default_rng(42)

background_indices = rng.choice(
    len(X_train_time),
    size=min(100, len(X_train_time)),
    replace=False
)

explain_indices = rng.choice(
    len(X_test_time),
    size=min(500, len(X_test_time)),
    replace=False
)

# 4. Apply the SAME fitted preprocessing used by XGBoost.
X_background = fitted_preprocessor.transform(
    X_train_time.iloc[background_indices]
)

X_explain = fitted_preprocessor.transform(
    X_test_time.iloc[explain_indices]
)

# Convert small samples to dense arrays if needed.
if sparse.issparse(X_background):
    X_background = X_background.toarray()

if sparse.issparse(X_explain):
    X_explain = X_explain.toarray()

# 5. Create the SHAP explainer.
explainer = shap.TreeExplainer(
    fitted_xgb,
    data=X_background,
    feature_perturbation="interventional",
    model_output="probability",
    feature_names=feature_names
)

# 6. Calculate SHAP values. This creates the missing variable.
shap_values = explainer(X_explain)

print("SHAP values calculated successfully!")
print("Explanation sample:", X_explain.shape)
print("SHAP values shape:", shap_values.values.shape)

In [ ]:
import numpy as np
import pandas as pd

# 1. Extract SHAP contribution values.
values = np.asarray(shap_values.values)

# Handle binary classification outputs if SHAP returns 3 dimensions.
if values.ndim == 3:
    values = values[:, :, 1]

if values.ndim != 2:
    raise ValueError(f"Unexpected SHAP array shape: {values.shape}")

# Verify feature names match the SHAP contributions.
if values.shape[1] != len(feature_names):
    raise ValueError("SHAP values and feature names do not match.")


# 2. Map transformed features back to original columns.
def original_feature_name(name):
    if name.startswith("numeric__missingindicator_"):
        return name.replace(
            "numeric__missingindicator_", "", 1
        )

    if name.startswith("numeric__"):
        return name.replace("numeric__", "", 1)

    if name.startswith("categorical__"):
        remainder = name.replace("categorical__", "", 1)

        # Match the longest categorical column name first.
        for column in sorted(
            categorical_features,
            key=len,
            reverse=True
        ):
            if remainder.startswith(column + "_"):
                return column

        return remainder

    return name


# 3. Combine contributions belonging to each original feature.
grouped_shap = {}

for i, transformed_name in enumerate(feature_names):
    original_name = original_feature_name(transformed_name)

    if original_name not in grouped_shap:
        grouped_shap[original_name] = np.zeros(values.shape[0])

    grouped_shap[original_name] += values[:, i]


# 4. Rank original features by their average absolute contribution.
importance_table = pd.DataFrame([
    {
        "original_feature": name,
        "mean_absolute_grouped_shap": np.mean(
            np.abs(contributions)
        )
    }
    for name, contributions in grouped_shap.items()
]).sort_values(
    "mean_absolute_grouped_shap",
    ascending=False
).reset_index(drop=True)


# 5. Display and save the results.
top_features = importance_table.head(10)

print("NOVABANK — TOP 10 MODEL FEATURES")
display(top_features)

top_features.to_csv(
    REPORT_DIR / "shap_top_10_features.csv",
    index=False
)

importance_table.to_csv(
    REPORT_DIR / "shap_global_importance_by_feature.csv",
    index=False
)

print("Feature-importance reports saved successfully.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse

# Recalculate risk scores on the temporal test set.
test_probabilities = xgb_model.predict_proba(
    X_test_time
)[:, 1]

# Select the highest-scored test loan based ONLY on its
# predicted risk, as would happen before the outcome is known.
case_position = int(np.argmax(test_probabilities))

case_input = X_test_time.iloc[[case_position]]
case_record_id = test.iloc[case_position]["record_id"]
case_actual_outcome = int(
    y_test_time.iloc[case_position]
    if hasattr(y_test_time, "iloc")
    else y_test_time[case_position]
)
case_predicted_probability = float(
    test_probabilities[case_position]
)

# Transform using the exact fitted preprocessing pipeline.
case_transformed = fitted_preprocessor.transform(case_input)

if sparse.issparse(case_transformed):
    case_transformed = case_transformed.toarray()

# Explain this individual observation.
case_explanation = explainer(case_transformed)

case_values = np.asarray(case_explanation.values)

if case_values.ndim == 3:
    case_values = case_values[0, :, 1]
else:
    case_values = case_values.reshape(-1)

print("Record ID:", case_record_id)
print(f"Model predicted probability: {case_predicted_probability:.4f}")
print(
    "Observed historical outcome:",
    "Charged Off" if case_actual_outcome == 1 else "Fully Paid"
)

In [ ]:
shap.plots.waterfall(
    case_explanation[0],
    max_display=12,
)

In [ ]:
local_table = pd.DataFrame({
    "feature": feature_names,
    "shap_contribution": case_values,
})

local_table["direction"] = np.select(
    [
        local_table["shap_contribution"] > 0,
        local_table["shap_contribution"] < 0,
    ],
    [
        "Pushes risk higher",
        "Pushes risk lower",
    ],
    default="No contribution",
)

local_table["absolute_contribution"] = (
    local_table["shap_contribution"].abs()
)

local_table = local_table.sort_values(
    "absolute_contribution",
    ascending=False,
).reset_index(drop=True)

display(local_table.head(10))

local_table.to_csv(
    REPORT_DIR / "shap_individual_case_contributions.csv",
    index=False,
)

In [ ]:
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

shap.plots.waterfall(
    case_explanation[0],
    max_display=12,
    show=False,
)

plt.title("NovaBank — Individual Risk Explanation")
plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "shap_individual_case.png",
    dpi=160,
    bbox_inches="tight",
)

plt.show()
plt.close()

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse


# 1. Recalculate scores on the temporal test set.
test_scores = xgb_model.predict_proba(X_test_time)[:, 1]

# Select the highest-scored case without using its actual outcome.
case_position = int(np.argmax(test_scores))
case_input = X_test_time.iloc[[case_position]]

case_score = float(test_scores[case_position])

# 2. Transform the case using the same fitted preprocessing.
case_transformed = fitted_preprocessor.transform(case_input)

if sparse.issparse(case_transformed):
    case_transformed = case_transformed.toarray()

# 3. Obtain SHAP values for this exact case.
case_explanation = explainer(case_transformed)

case_values = np.asarray(case_explanation.values)

if case_values.ndim == 3:
    case_values = case_values[0, :, 1]
else:
    case_values = case_values.reshape(-1)

if len(case_values) != len(feature_names):
    raise ValueError("SHAP values do not match model feature names.")


# 4. Group encoded contributions under original feature names.
def original_feature_name(name):
    if name.startswith("numeric__missingindicator_"):
        return name.replace("numeric__missingindicator_", "", 1)

    if name.startswith("numeric__"):
        return name.replace("numeric__", "", 1)

    if name.startswith("categorical__"):
        remainder = name.replace("categorical__", "", 1)

        for column in sorted(categorical_features, key=len, reverse=True):
            if remainder.startswith(column + "_"):
                return column

        return remainder

    return name


grouped = {}

for index, transformed_name in enumerate(feature_names):
    original_name = original_feature_name(transformed_name)

    grouped[original_name] = (
        grouped.get(original_name, 0.0)
        + float(case_values[index])
    )


# 5. Create a readable evidence table.
def safe_value(value):
    if pd.isna(value):
        return None

    if isinstance(value, np.generic):
        return value.item()

    return value


feature_evidence = []

for feature, contribution in grouped.items():
    observed_value = (
        safe_value(case_input.iloc[0][feature])
        if feature in case_input.columns
        else None
    )

    if contribution > 0:
        direction = "increases_model_score"
    elif contribution < 0:
        direction = "decreases_model_score"
    else:
        direction = "no_material_direction"

    feature_evidence.append({
        "feature": feature,
        "observed_value": observed_value,
        "shap_contribution": round(contribution, 8),
        "direction": direction,
    })

feature_evidence.sort(
    key=lambda item: abs(item["shap_contribution"]),
    reverse=True,
)

# Use the top eight grouped features in the first explanation.
top_evidence = feature_evidence[:8]

base_value = float(
    np.asarray(case_explanation.base_values).reshape(-1)[0]
)

reconstructed_score = base_value + sum(grouped.values())


# 6. Save the evidence packet.
PROJECT_ROOT = Path(
    r"C:\Users\DELL\OneDrive\Desktop\NovaBank"
)

COPILOT_DIR = (
    PROJECT_ROOT / "reports" / "credit_risk" / "ai_copilot"
)
COPILOT_DIR.mkdir(parents=True, exist_ok=True)

evidence_packet = {
    "case_type": "historical_temporal_test_example",
    "model": "XGBoost",
    "model_score": round(case_score, 6),
    "score_interpretation": (
        "Uncalibrated model-predicted probability; "
        "not a validated individual default probability."
    ),
    "shap_base_value": round(base_value, 8),
    "shap_reconstructed_score": round(reconstructed_score, 6),
    "feature_evidence": top_evidence,
    "limitations": [
        "Historical US lending data from 2007–2011.",
        "The model was evaluated on resolved loans only.",
        "SHAP contributions describe model behaviour, not causation.",
        "Probability calibration has not been established as adequate.",
        "This is a retrospective portfolio demonstration, not a lending decision."
    ],
}

packet_path = COPILOT_DIR / "evidence_packet.json"

with open(packet_path, "w", encoding="utf-8") as file:
    json.dump(evidence_packet, file, indent=2, allow_nan=False)

print("Evidence packet saved:", packet_path)
print("Model score:", case_score)
print("SHAP reconstructed score:", reconstructed_score)

display(pd.DataFrame(top_evidence))

In [ ]:
import json
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(
    r"C:\Users\DELL\OneDrive\Desktop\NovaBank"
)

COPILOT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "credit_risk"
    / "ai_copilot"
)

# Load the evidence supplied to the LLM.
with open(
    COPILOT_DIR / "evidence_packet.json",
    encoding="utf-8"
) as f:
    evidence = json.load(f)

# Load the explanation produced by the LLM.
with open(
    COPILOT_DIR / "risk_explanation.json",
    encoding="utf-8"
) as f:
    explanation = json.load(f)

print("MODEL SCORE:", evidence["model_score"])
print("SHAP RECONSTRUCTED SCORE:",
      evidence["shap_reconstructed_score"])

print("\n=== VERIFIED FEATURE EVIDENCE ===")

evidence_table = pd.DataFrame(
    evidence["feature_evidence"]
)

display(evidence_table)

print("\n=== AI GENERATED SUMMARY ===")
print(explanation["summary"])

print("\n=== AI SUGGESTED ANALYST CHECKS ===")
for check in explanation["analyst_checks"]:
    print("-", check)